# Changes from ExRD, and how to read Backward Transfer**Alexie Linardatos**This covers three things: exactly what was changed in the codebase relative to the ExRDbaseline, what Backward Transfer measures and how it is computed, and the results table.

## 1. The baselineExRD is the phase-one method. ZSCL distillation against the original pre-trained CLIP, plus areplay-teacher distillation term against the previous task's model, plus supervised replaycross-entropy, plus an L2 anchor. Five loss terms:$$L = L_{CE} + L_{zscl} + L_{RD} + L_{rep} + L_{2}$$Its replay buffer holds **11,000 preprocessed images**. Each is a 224 x 224 x 3 float32 tensor,602 kB, so the buffer is **6.62 GB**. No MTIL paper reports this number, and that is what phasetwo set out to attack.Baseline results: Avg 77.17, Transfer 69.18, Last 86.17.

## 2. What changed: one flagEverything in phase two sits behind a single argument on the existing trainer:> --replay_storage {pixel, feature, remind}Nothing else about training was touched. Explicitly unchanged:| | Status ||---|---|| All five loss terms | unchanged || Optimizer (AdamW), learning rate 5e-6, label smoothing | unchanged || Weight averaging, avg_freq 50 | unchanged || L2 anchor weight | unchanged || ZSCL reference dataset and sentences | unchanged || Task order and per-task iteration schedule | unchanged || Replay budget (11,000) and batch size (8) | unchanged || Evaluation protocol | unchanged |This matters for a specific reason. Because pixel mode is bit-identical to ExRD, every row in theresults table differs from the baseline by exactly one factor, so nothing in the comparison isconfounded by a hyperparameter change.

## 3. What was added### New files| File | Lines | Contents ||---|---|---|| src/feature_replay_buffer.py | 343 | stores the final 512-d embedding per exemplar || src/feature_adaptation.py | ~500 | the drift-correction ladder (SDC, label propagation, learned maps) || src/product_quantizer.py | 201 | k-means product quantization, written from scratch to avoid a faiss dependency the cluster may not carry || src/remind_buffer.py | 493 | the split forward pass, the layer freeze, and the REMIND buffer || src/drift_probe.py | 194 | diagnostic only, never enters a loss |For scale, the original pixel buffer is 152 lines.### Modified files| File | Change | Size ||---|---|---|| phase3/losses_phase3.py | two new loss functions, one per storage mode | ~130 lines || phase3/trainer_phase3.py | one dispatch dictionary, one freeze call, one requires_grad filter | ~25 lines || phase3/finetune_phase3.py | buffer construction branch, end-of-task encoding branch | ~60 lines || phase3/args_phase3.py | the flags | ~80 lines |The dispatch is three lines and is the entire integration point:> replay_ce_fn = {"feature": compute_feature_replay_loss, "remind": compute_remind_replay_loss}.get(replay_storage, compute_replay_loss)

## 4. The three storage modes| | Pixel (ExRD) | Feature replay | REMIND ||---|---|---|---|| What is stored | 224x224x3 image tensor | final 512-d embedding, fp16 | PQ-compressed mid-network token grid || Bytes per exemplar | 602 kB | 1.02 kB | 6.3 kB at m=32, 25.2 kB at m=128 || Buffer at 11,000 | 6.62 GB | 11.3 MB | 69 MB to 277 MB || Gradient reaches | the whole vision tower and the text tower | the text tower only | the vision blocks above the tap, and the text tower || Representation drift | none, pixels do not change | full, all 12 blocks sit below the storage point | none, everything below the tap is frozen || Augmentation possible | yes, fresh crop every epoch | no, the vector is fixed | not implemented |

## 5. What REMIND adds specificallyThree mechanisms, all from Hayes et al., ECCV 2020.**A split forward pass.** Two functions carve CLIP's vision transformer in half. encode_to_layerruns the patch embedding and the first k residual blocks and returns the 197 x 768 token grid.encode_from_layer takes a stored grid and runs blocks k onward plus ln_post and the projection.Together they reproduce VisualTransformer.forward exactly, which had to be verified step by step:class token, positional embedding, ln_pre, the NLD to LND permutes, ln_post applied only to tokenzero, and the final projection.**Product quantization.** A raw token grid is 605 kB, larger than the 602 kB image it came from,so compression is not an optimization here, it is what makes mid-network storage viable at all.Each 768-dimensional token is split into m contiguous chunks, each chunk is replaced by the indexof the nearest of 256 learned centroids, and the token costs m bytes.**Freezing below the tap.** The layers that produced the stored activations never change again, sothe activations cannot go stale. This is drift prevention rather than drift correction, and it isthe reason REMIND was the right paper to build on after the correction ladder failed.Only the **image encoder** is frozen. The text encoder trains throughout, which matters becausereplay cross-entropy is the only term in the objective that updates it for previously seen classes.| Tap | Frozen parameters | Share of vision tower | Share of whole model ||---|---|---|---|| block 2 | 14.9M | 17% | 10% || block 6 | 43.3M | 50% | 29% |

## 6. Two adaptations REMIND does not haveBoth address places where the original's assumptions are false for CLIP. Both are on by defaultwith ablation flags.**Per-task codebooks** (disable with remind_shared_codebook). REMIND fits one compressiondictionary on the first task and reuses it forever, which is sound on a single-domain ImageNetstream. On MTIL it meant a dictionary built from aircraft photographs was compressing MNIST digitsand satellite imagery. Each task now fits and keeps its own, and decodes with the one it wasencoded by, so a later fit cannot corrupt an earlier code.**Per-channel standardisation** (disable with remind_no_standardize). Transformer residual streamscarry a handful of channels an order of magnitude larger than the rest, which dominate whicheverquantization chunks they land in. Each task's tokens are now centred and scaled per dimensionbefore quantizing and un-scaled on decode.Honest note on the second: this is standardisation, not whitening. Whitening removes correlationbetween dimensions, which is what contiguous-chunk quantization actually needs. Standardisationonly equalises their variance. Measured effect on the 11-task run was negative.Two diagnostic arms were also added, remind_no_pq (store fp16 tokens with no quantization, as anoracle) and remind_no_freeze.

## 7. Backward Transfer### What it measuresLast conflates two different things. A method can score badly because it never learned the taskwell, or because it learned it and then forgot. Backward Transfer separates them.Evaluation produces a matrix. Call it A, where> **A[i][j] = accuracy on task j, measured after training task i**Two parts of it matter:- **The diagonal, A[j][j].** Accuracy on task j immediately after learning it, before anything  else interferes. This is **plasticity**: how well the model could learn that task at all.- **The final row, A[T][j].** Accuracy on task j after every task is done. This is **Last**.Backward Transfer is the average gap between them:$$\mathrm{BWT} = \frac{1}{T-1}\sum_{j=1}^{T-1} \Big( A[T][j] - A[j][j] \Big)$$Negative means forgetting. Positive means later tasks improved earlier ones.The last task is excluded because it was trained last, so its diagonal entry and its final entryare the same cell and the difference is zero by construction.### A shortcutBecause that zero contributes nothing, BWT can be computed from two aggregate numbers:$$\mathrm{BWT} = \frac{T}{T-1}\Big( \mathrm{Last} - \mathrm{Diagonal} \Big)$$For the 11-task benchmark that is 1.1 times (Last minus Diagonal).

### Worked exampleA four-task run. Rows are the stage after training each task, columns are what was evaluated.| After training | A | B | C | D ||---|---|---|---|---|| **A** | **90** | 40 | 30 | 50 || **B** | 86 | **88** | 34 | 52 || **C** | 82 | 85 | **76** | 55 || **D** | 80 | 83 | 72 | **94** |Diagonal in bold. The final row is the bottom one.| Task | Diagonal | Final | Difference ||---|---|---|---|| A | 90 | 80 | -10 || B | 88 | 83 | -5 || C | 76 | 72 | -4 || D | 94 | 94 | 0, excluded |$$\mathrm{BWT} = \frac{-10 - 5 - 4}{3} = -6.33$$Checking against the shortcut: the diagonal mean is 87.00, Last is 82.25, and(4/3) x (82.25 - 87.00) = -6.33. Same answer.Task A forgets most, having sat through three subsequent tasks. Task D forgets nothing becausenothing came after it.### Why it must be reported alongside the diagonalConsider a completely frozen model. It forgets nothing, so BWT is exactly zero, a perfect score.Its Last is 65.3, which is CLIP's zero-shot accuracy, because it never learned anything. It wouldtop a BWT leaderboard and be useless.The opposite extreme, plain fine-tuning, has a high diagonal and catastrophic forgetting.Neither number means anything alone. Together they say which half of the problem a method solves.

## 8. ResultsSingle seed, 11-task MTIL Order-I, CLIP ViT-B/16. Transfer is the upper triangle, the conventionZSCL, DIKI, BCL and AFA report.| Run | Avg | Transfer | Last | BWT | Buffer ||---|---|---|---|---|---|| **ExRD / v4** (pixel replay) | **77.17** | **69.18** | **86.17** | -1.08 | 6.6 GB || **Feature replay v0** (embeddings) | 76.08 | 68.81 | 84.79 | -2.13 | 11 MB || **REMIND tap 6, m=128** | 76.83 | 69.10 | 85.88 | **-0.55** | 277 MB |### Reading it**Feature replay is the cheap extreme.** 588x less storage, and it pays 1.38 points of Last anddoubles the forgetting.**REMIND at m=128 forgets half as much as storing the real images**, -0.55 against -1.08, at 24xless storage. It is the best retention of anything tested, including the 6.6 GB baseline.**Its deficit is plasticity, not retention.** The diagonals are 87.15 for ExRD and 86.38 forREMIND, so REMIND learned each task 0.77 points worse in the first place. That is the frozenblocks, and it is the entire remaining gap. Reported as Last alone this reads as "REMIND isworse." Reported with BWT it reads as "REMIND has the best retention and pays for it inplasticity," which names the next thing to change.The lever for plasticity is the tap depth. At block 2 the freeze covers 14.9M parameters insteadof 43.3M, and at m=32 a block-2 tap already produced the highest diagonal of any REMIND run.### One caveat on the Transfer columncompute_metrics.py computes the upper triangle by pooling all 55 cells. AFA computes it as themean of the ten per-dataset column means, which is a different number, because later tasks havemore rows and so carry more weight when pooled. 69.18 is the right quantity for comparing againstthe literature but may not be the right arithmetic, and that should be checked before thesenumbers go into a table next to published ones.